# Etapa 4 en Colab: fine-tuning parcial con aumentación dirigida (sesión 10)

Lo ejecuta el usuario, **en orden**, con entorno de ejecución **GPU (T4)**. Decisiones a–h: `docs/bitacora/decisiones.md` y §f de `docs/protocolo_evaluacion.md`.

**Antes de empezar (una sola vez):**
1. Subí el checkpoint del modelo base de la PC (`C:\Users\xbetr\foliares_salidas\02_modelo_base\02_modelo_base_20261104_201229\mejor_checkpoint.pt`) a Drive en `foliares/checkpoints/02_modelo_base/02_modelo_base_20261104_201229/mejor_checkpoint.pt`. El script verifica su sha256 contra la config.
2. En la celda 2 poné `REPO_REF` = la rama de trabajo (`sesion-10/aumentacion`), pusheada y con el árbol limpio.

**Qué hace:** prepara los datos como `colab_arranque.ipynb` pero **solo con el paquete de PlantVillage** (PlantDoc no entra a Colab: no entrena ni elige), entrena con `configs/04_aumentacion.yaml` y deja el checkpoint de cada época en `foliares/checkpoints/04_aumentacion/<run_id>/` y el JSON en `foliares/results/04_aumentacion/<run_id>/`.

**Si la sesión se corta:** abrí una sesión nueva, corré otra vez las celdas 1–9 con `RUN_ID_RETOMAR` = el run_id que se imprimió, y la celda 10 retoma desde la última época completa.

**Al terminar:** pegá la salida de la celda 11 y bajá a la PC la carpeta `foliares/checkpoints/04_aumentacion/<run_id>/` (al menos `mejor.pt`) y `foliares/results/04_aumentacion/<run_id>/resultados.json` en `C:\Users\xbetr\foliares_salidas\04_aumentacion\<run_id>\`.

In [ ]:
# 1) Montar Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 2) Variables de la sesión y utilidades
import os, subprocess, time, json
DRIVE_DIR = "/content/drive/MyDrive/foliares"       # carpeta compartida del equipo (acceso directo en Mi unidad)
REPO_URL  = "https://github.com/trajano-carballo/enfermedades-foliares"   # público: se clona sin token
# Rama, tag o commit a clonar. En cada sesión se cambia en la interfaz de Colab (campo del formulario) a la rama de trabajo,
# sin commitear ese cambio: en el repo queda "main". El JSON de sesión ya registra el commit clonado.
REPO_REF  = "main"  # @param {type:"string"}
REPO_DIR  = "/content/enfermedades-foliares"      # NO llamarlo `foliares`: chocaría con el nombre del paquete
DATA_ROOT = "/content/data"                         # destino de los .tar (disco local de Colab, rápido)
PAQUETES_DRIVE = f"{DRIVE_DIR}/paquetes"            # solo se usa plantvillage_subconjunto.tar(.sha256)
CKPT_DIR  = f"{DRIVE_DIR}/checkpoints"              # checkpoints: SIEMPRE en Drive
SALIDA    = f"{DRIVE_DIR}/results"                  # results/<experimento>/<run_id>/ (JSON de cada corrida), en Drive
RUN_ID    = time.strftime("%Y%m%d_%H%M%S")          # identifica esta sesión en results/ y checkpoints/
T = {}                                              # tiempos de la preparación (s), van al JSON de sesión

assert os.path.isdir(PAQUETES_DRIVE), f"No existe {PAQUETES_DRIVE}: ¿está el acceso directo 'foliares' en Mi unidad?"
os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(SALIDA, exist_ok=True)

def sh(cmd, check=True):
    """Corre un comando de shell, muestra su salida y devuelve el código de salida."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr, end="")
    if check and r.returncode:
        raise RuntimeError(f"falló ({r.returncode}): {cmd}")
    return r.returncode

print(sh("nvidia-smi -L", check=False) == 0 and "GPU disponible" or "SIN GPU: cambiar el tipo de entorno de ejecución")

# --- Entrenamiento de la etapa 4 (sesión 10) ---
# Para RETOMAR una corrida cortada, poné acá su run_id (el que imprimió la celda 10); vacío = corrida nueva.
RUN_ID_RETOMAR = ""  # @param {type:"string"}
RUN_ENTRENAMIENTO = RUN_ID_RETOMAR.strip() or f"04_aumentacion_{RUN_ID}"
CKPT_BASE = f"{CKPT_DIR}/02_modelo_base/02_modelo_base_20261104_201229/mejor_checkpoint.pt"   # subirlo antes a Drive
print("run de entrenamiento:", RUN_ENTRENAMIENTO, "| retomar:", bool(RUN_ID_RETOMAR.strip()))

In [ ]:
# 3) Clonar el repo en el ref indicado
t0 = time.perf_counter()
if os.path.isdir(f"{REPO_DIR}/.git"):
    sh(f"git -C {REPO_DIR} fetch -q origin")           # celda re-ejecutada: no se vuelve a clonar
else:
    sh(f"git clone -q {REPO_URL} {REPO_DIR}")
sh(f"git -C {REPO_DIR} checkout -q {REPO_REF}")
os.chdir(REPO_DIR)
T["clonar_s"] = round(time.perf_counter() - t0, 1)
sh("git log --oneline -1")

In [ ]:
# 4) Instalar el paquete (torch/torchvision ya vienen en Colab) y hacerlo visible en ESTE kernel, sin reiniciar
import sys, importlib
t0 = time.perf_counter()
sh("pip install -q -e . --no-deps && pip install -q imagehash pytest")
T["pip_s"] = round(time.perf_counter() - t0, 1)

# Un `pip install -e` no se ve en el kernel ya iniciado (el .pth del editable se procesa al arrancar el intérprete), y un
# directorio llamado `foliares` en sys.path se importaría como paquete namespace vacío ("No module named 'foliares.utils'").
# Por eso `src` va primero en sys.path (y en PYTHONPATH, para los subprocesos).
SRC = f"{REPO_DIR}/src"
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.environ["PYTHONPATH"] = os.pathsep.join([SRC] + [p for p in os.environ.get("PYTHONPATH", "").split(os.pathsep) if p and p != SRC])
for nombre in [m for m in sys.modules if m == "foliares" or m.startswith("foliares.")]:
    del sys.modules[nombre]                  # descarta un `foliares` vacío cacheado por un import fallido
importlib.invalidate_caches()
import foliares
print("foliares.__file__:", foliares.__file__)
assert foliares.__file__ and os.path.realpath(foliares.__file__).startswith(os.path.realpath(SRC)),     f"`foliares` no sale de {SRC}: __file__={foliares.__file__!r}, __path__={list(foliares.__path__)}"

In [ ]:
# 5) Copiar los .tar desde Drive y descomprimirlos en DATA_ROOT (nombres dentro del tar = `ruta` de los manifiestos)
import shutil, tarfile
os.makedirs(DATA_ROOT, exist_ok=True); os.makedirs("/content/paquetes", exist_ok=True)
t0 = time.perf_counter(); tars = []
for ds in ("plantvillage",):          # PlantDoc no hace falta para entrenar (invariante 1)
    for ext in (".tar", ".tar.sha256"):
        shutil.copy(f"{PAQUETES_DRIVE}/{ds}_subconjunto{ext}", "/content/paquetes/")
    tars.append(f"/content/paquetes/{ds}_subconjunto.tar")
T["copiar_s"] = round(time.perf_counter() - t0, 1)
t0 = time.perf_counter()
for t in tars:
    with tarfile.open(t) as tf:
        tf.extractall(DATA_ROOT, filter="data")
    print("descomprimido:", t)
T["descomprimir_s"] = round(time.perf_counter() - t0, 1)
print(T)

In [ ]:
# 6) Fijar DATA_ROOT (lo lee configs/paths.yaml) y verificar sha256 del tar y de cada archivo
os.environ["DATA_ROOT"] = DATA_ROOT
t0 = time.perf_counter()
rc_verificar = sh(f"python scripts/verificar_paquete.py --tar {' '.join(tars)}", check=False)
T["verificar_s"] = round(time.perf_counter() - t0, 1)
print("verificar_paquete.py -> código de salida:", rc_verificar)
assert rc_verificar == 0, "Hay MAL en la verificación: NO seguir; volver a copiar los .tar desde Drive."
print(T)

In [ ]:
# 7) Particiones congeladas (viven en el repo, no en los .tar) y commit clonado vs. rama remota
from foliares.utils.paths import cargar_rutas
R = cargar_rutas()
print("DATA_ROOT:", R.data_root, "| SPLITS_DIR:", R.splits)
assert (R.splits / "README_VERSION.md").exists(), "Faltan las particiones congeladas en el repo"
assert sh(f"cd {R.splits} && sha256sum -c SHA256SUMS.txt", check=False) == 0, "data/splits/ no coincide con SHA256SUMS.txt"
commit_local = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
ls = subprocess.check_output(["git", "-C", REPO_DIR, "ls-remote", "origin", REPO_REF], text=True).split()
commit_rama = ls[0] if ls else None                 # None si REPO_REF es un commit y no una rama/tag
print("commit clonado:", commit_local, "| commit de la rama remota:", commit_rama, "| coinciden:", commit_local == commit_rama)

In [ ]:
# 8) JSON de sesión (en Drive): commit, splits, semillas, torch, GPU, tiempos de la preparación
import torch
from foliares.utils.seeds import fijar_semillas
from foliares.utils.sesion import registro_base, escribir_json
sesion = registro_base(REPO_DIR, R.splits, {"repo_url": REPO_URL, "repo_ref": REPO_REF, "data_root": DATA_ROOT}, fijar_semillas(42))
sesion.update({"run_id": RUN_ID, "verificar_paquete_codigo": rc_verificar, "commit_rama_remota": commit_rama,
               "commit_coincide_con_la_rama": commit_local == commit_rama, "tiempos_preparacion_s": T,
               "tiempo_preparacion_total_s": round(sum(T.values()), 1)})
ruta_sesion = escribir_json(f"{SALIDA}/colab_sesion/{RUN_ID}/sesion.json", sesion)
print(json.dumps({k: sesion[k] for k in ("commit", "cambios_sin_commitear", "torch", "cuda_runtime", "cudnn", "gpu", "python",
                                         "verificar_paquete_codigo", "commit_coincide_con_la_rama", "tiempos_preparacion_s",
                                         "tiempo_preparacion_total_s")}, indent=2))
print("guardado en:", ruta_sesion)

In [ ]:
# 9) Checkpoint del modelo base en Drive (lo verifica el script por sha256)
import yaml
from foliares.data.congelar import sha256_archivo
assert os.path.isfile(CKPT_BASE), f"Falta {CKPT_BASE}: subí el checkpoint del modelo base a Drive (ver la celda de arriba)"
esperado = yaml.safe_load(open("configs/04_aumentacion.yaml", encoding="utf-8"))["modelo_base"]["sha256"]
print("sha256 del checkpoint base correcto:", sha256_archivo(CKPT_BASE) == esperado)
assert sha256_archivo(CKPT_BASE) == esperado

In [ ]:
# 10) Entrenar (o retomar). La salida se muestra época por época.
cmd = ["python", "scripts/entrenar_finetune.py", "--config", "configs/04_aumentacion.yaml", "--salida", SALIDA,
       "--ckpt-dir", CKPT_DIR, "--checkpoint-base", CKPT_BASE, "--run-id", RUN_ENTRENAMIENTO]
print(" ".join(cmd), flush=True)
t0 = time.perf_counter()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for linea in proc.stdout:
    print(linea, end="", flush=True)
rc_entrenamiento = proc.wait()
print(f"\ncódigo de salida: {rc_entrenamiento} | {round(time.perf_counter() - t0)} s | run: {RUN_ENTRENAMIENTO}")

In [ ]:
# 11) RESUMEN PARA PEGAR en la conversación (copiar toda la salida de esta celda)
r = json.load(open(f"{SALIDA}/04_aumentacion/{RUN_ENTRENAMIENTO}/resultados.json"))
resumen = {k: r[k] for k in ("run_id", "commit", "cambios_sin_commitear", "torch", "cuda_runtime", "cudnn", "gpu", "dispositivo",
                             "n_train", "n_val", "parametros_entrenables", "epoca_elegida", "epocas_corridas", "detenido_por",
                             "reanudaciones_desde_epoca", "tiempo_s", "memoria_pico_gpu_mb", "memoria_pico_cpu_mb", "checkpoint_mejor")}
resumen["macro_f1_pv_val"] = r["pv_val"]["macro_f1"]
resumen["curva"] = [{k: (round(v, 4) if isinstance(v, float) else v) for k, v in e.items()} for e in r["curva"]]
print(json.dumps(resumen, indent=2, ensure_ascii=False))
print("\nArchivos en Drive para bajar a la PC:", sorted(os.listdir(f"{CKPT_DIR}/04_aumentacion/{RUN_ENTRENAMIENTO}")))